In [6]:
import json
import random
import shutil
import unicodedata
from pathlib import Path
from collections import Counter
from sklearn.model_selection import StratifiedKFold

SEED = 42
N_FOLDS = 5
QUARANTINE_IDS = {103, 104, 105, 108, 116, 121, 136, 141, 457}

DATA_PATH = Path(
    "/kaggle/input/datasets/tuandatdao/vgec2026-bartpho/VGEC 2026_v2.json"
)
SAVE_DIR = Path("/kaggle/working/vgec_preprocessed")

for folder in ["raw", "processed", "quarantine", "folds"]:
    (SAVE_DIR / folder).mkdir(parents=True, exist_ok=True)

def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def normalize(text):
    # Chỉ chuẩn hóa Unicode, giữ cách viết, dấu câu và khoảng trắng.
    return unicodedata.normalize("NFC", text)

with open(DATA_PATH, "r", encoding="utf-8-sig") as f:
    raw_data = json.load(f)

# 1. Lưu nguyên bản raw.
shutil.copy2(DATA_PATH, SAVE_DIR / "raw" / DATA_PATH.name)

# 2. Tách 9 ID khỏi cả train và validation.
quarantine = [
    row for row in raw_data
    if row["ID"] in QUARANTINE_IDS
]
save_json(SAVE_DIR / "quarantine" / "records.json", quarantine)

# 3. Normalize, tạo identity pairs và gộp input trùng.
groups = {}

for row in raw_data:
    if row["ID"] in QUARANTINE_IDS:
        continue

    source = normalize(row["Input"])
    label = row["Output"]["Error_Type_Id"]

    if label == 0:
        references = [source]
    else:
        references = [
            normalize(ref)
            for ref in row["Output"]["Fixed Sentence"]
        ]

    if source not in groups:
        groups[source] = {
            "id": row["ID"],
            "original_ids": [],
            "source": source,
            "error_type": label,
            "references": []
        }

    sample = groups[source]
    assert sample["error_type"] == label, (
        f"Input trùng nhưng khác nhãn: {sample['original_ids']} và {row['ID']}"
    )

    sample["original_ids"].append(row["ID"])

    for ref in references:
        if ref not in sample["references"]:
            sample["references"].append(ref)

data = sorted(groups.values(), key=lambda sample: sample["id"])

# 4. Chia stratified 5-fold.
# Mỗi input đã được gộp thành đúng một sample,
# nên split ở đây cũng bảo đảm các input trùng không qua hai fold.
splitter = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

labels = [sample["error_type"] for sample in data]

for fold, (_, val_indices) in enumerate(
    splitter.split(data, labels)
):
    for index in val_indices:
        data[index]["fold"] = fold

save_json(SAVE_DIR / "processed" / "all.json", data)

# 5. Xuất train/validation cho từng fold.
fold_stats = []

for fold in range(N_FOLDS):
    fold_dir = SAVE_DIR / "folds" / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    train_data = [sample for sample in data if sample["fold"] != fold]
    val_data = [sample for sample in data if sample["fold"] == fold]

    train_sources = {sample["source"] for sample in train_data}
    val_sources = {sample["source"] for sample in val_data}
    assert train_sources.isdisjoint(val_sources)

    save_json(fold_dir / "train.json", train_data)
    save_json(fold_dir / "val.json", val_data)

    fold_stats.append({
        "fold": fold,
        "train_size": len(train_data),
        "val_size": len(val_data),
        "train_labels": dict(Counter(
            sample["error_type"] for sample in train_data
        )),
        "val_labels": dict(Counter(
            sample["error_type"] for sample in val_data
        ))
    })

# 6. Lưu thông tin xử lý và mapping truy ngược ID gốc.
metadata = {
    "seed": SEED,
    "n_folds": N_FOLDS,
    "normalization": "Unicode NFC only",
    "raw_size": len(raw_data),
    "quarantine_size": len(quarantine),
    "quarantine_ids": sorted(QUARANTINE_IDS),
    "duplicates_merged": len(raw_data) - len(quarantine) - len(data),
    "processed_size": len(data),
    "label_counts": dict(Counter(
        sample["error_type"] for sample in data
    )),
    "folds": fold_stats
}

save_json(SAVE_DIR / "processed" / "metadata.json", metadata)

save_json(
    SAVE_DIR / "processed" / "id_mapping.json",
    [
        {
            "id": sample["id"],
            "original_ids": sample["original_ids"],
            "fold": sample["fold"]
        }
        for sample in data
    ]
)

# 7. Đóng gói để tải về.
zip_path = shutil.make_archive(
    str(SAVE_DIR),
    "zip",
    root_dir=SAVE_DIR
)

print("Số mẫu raw:", len(raw_data))
print("Số mẫu quarantine:", len(quarantine))
print("Số bản ghi trùng được gộp:", metadata["duplicates_merged"])
print("Số mẫu sau preprocessing:", len(data))
print("Phân bố nhãn:", metadata["label_counts"])

for stats in fold_stats:
    print(
        f"Fold {stats['fold']}: "
        f"train={stats['train_size']}, "
        f"val={stats['val_size']} | "
        f"Nhãn val: {stats['val_labels']}"
    )

print("\nĐã lưu:", SAVE_DIR)
print("File ZIP:", zip_path)

Số mẫu raw: 457
Số mẫu quarantine: 9
Số bản ghi trùng được gộp: 7
Số mẫu sau preprocessing: 441
Phân bố nhãn: {1: 146, 2: 28, 3: 31, 4: 36, 0: 200}
Fold 0: train=352, val=89 | Nhãn val: {1: 30, 2: 5, 3: 6, 4: 8, 0: 40}
Fold 1: train=353, val=88 | Nhãn val: {1: 29, 2: 6, 3: 6, 4: 7, 0: 40}
Fold 2: train=353, val=88 | Nhãn val: {1: 29, 2: 6, 3: 6, 4: 7, 0: 40}
Fold 3: train=353, val=88 | Nhãn val: {1: 29, 2: 6, 3: 6, 4: 7, 0: 40}
Fold 4: train=353, val=88 | Nhãn val: {1: 29, 2: 5, 3: 7, 4: 7, 0: 40}

Đã lưu: /kaggle/working/vgec_preprocessed
File ZIP: /kaggle/working/vgec_preprocessed.zip


In [7]:
%pip install -q "transformers==4.57.1" "accelerate==1.10.1" sentencepiece protobuf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 82.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 8.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 15.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 74.4 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import json
import math
import random
from pathlib import Path

import torch
import transformers
from packaging.version import Version
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback,
    EarlyStoppingCallback,
    set_seed,
)

assert torch.cuda.is_available(), "M cần bật GPU trước."
assert Version(torch.__version__.split("+")[0]) >= Version("2.6"), (
    "Checkpoint gốc dùng .bin và cần PyTorch >= 2.6 với Transformers này. "
    "Gửi t phiên bản torch đang dùng để chỉnh môi trường."
)

MODEL_NAME = "vinai/bartpho-syllable-base"
DATA_DIR = Path("/kaggle/working/vgec_preprocessed")
OUTPUT_DIR = Path("/kaggle/working/bartpho_vgec_joint")

SEED = 42
EPOCHS = 15
LEARNING_RATE = 3e-5
TRAIN_BATCH_SIZE = 2
GRAD_ACCUMULATION = 8
EVAL_BATCH_SIZE = 2

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
set_seed(SEED)

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)

def read_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def write_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

GPU: Tesla T4
PyTorch: 2.11.0+cu128
Transformers: 4.57.1


config.json:   0%|          | 0.00/898 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

dict.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
from torch.utils.data import Dataset
from transformers import TrainerCallback
def make_joint_target(label, correction):
    return (
        f"error_type: {label} "
        f"corrected_sentence: {correction}"
    )
class CorrectionDataset(Dataset):
    def __init__(self, records, tokenizer, training=False):
        self.records = records
        self.training = training
        self.epoch = 0

        # Tokenize trước toàn bộ input và các reference.
        self.sources = [
            tokenizer(row["source"], truncation=False)
            for row in records
        ]

        self.targets = [
            [
                tokenizer(
                    text_target=make_joint_target(
                        row["error_type"],
                        ref,
                    ),
                    truncation=False,
                )["input_ids"]
                for ref in row["references"]
            ]
            for row in records
        ]

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        refs = self.targets[index]

        if self.training:
            rng = random.Random(
                SEED + self.epoch * 1_000_003 + index
            )
            target = rng.choice(refs)
        else:
            target = refs[0]

        return {
            "input_ids": list(self.sources[index]["input_ids"]),
            "attention_mask": list(
                self.sources[index]["attention_mask"]
            ),
            "labels": list(target),
        }


class ReferenceEpochCallback(TrainerCallback):
    def __init__(self, dataset):
        self.dataset = dataset

    def on_epoch_begin(self, args, state, control, **kwargs):
        self.dataset.epoch = int(state.epoch or 0)

In [27]:
import re

@torch.inference_mode()
def predict_records(model, records, batch_size=2, max_length=256):
    model.eval()
    device = next(model.parameters()).device
    predictions = []
    limit = model.config.max_position_embeddings

    for start in range(0, len(records), batch_size):
        batch = records[start:start + batch_size]

        inputs = tokenizer(
            [row["source"] for row in batch],
            return_tensors="pt",
            padding=True,
            truncation=False,
        ).to(device)

        if inputs["input_ids"].shape[1] > limit:
            raise ValueError(
                f"Input vượt giới hạn {limit} tokens."
            )

        generated = model.generate(
            **inputs,
            num_beams=4,
            do_sample=False,
            max_length=min(max_length, limit),
            early_stopping=True,
            use_cache=True,
        )

        predictions.extend(
            tokenizer.batch_decode(
                generated,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            )
        )

    return predictions


def parse_joint_output(raw_text):
    # Nhận cả error_type và errortype.
    label_match = re.match(
        r"\A\s*error_?type\s*:\s*(-?\d+)(?=\s|$)",
        raw_text,
        flags=re.IGNORECASE,
    )

    # Nhận cả corrected_sentence và correctedsentence.
    correction_match = re.search(
        r"corrected_?sentence\s*:[ \t]?(.*)\Z",
        raw_text,
        flags=re.IGNORECASE | re.DOTALL,
    )

    label = (
        int(label_match.group(1))
        if label_match else -1
    )

    if label not in range(5):
        label = -1

    correction = (
        correction_match.group(1)
        if correction_match else raw_text
    )

    full_match = re.fullmatch(
        r"\s*error_?type\s*:\s*(-?\d+)\s+"
        r"corrected_?sentence\s*:[ \t]?(.*)",
        raw_text,
        flags=re.IGNORECASE | re.DOTALL,
    )

    return {
        "raw_prediction": raw_text,
        "predicted_error_type": label,
        "prediction": correction,
        "format_valid": bool(
            full_match and label != -1 and correction
        ),
    }


# Kiểm tra hai kiểu tên trường.
for text in [
    "error_type: 1 corrected_sentence: Tôi đang học.",
    "errortype: 1 correctedsentence: Tôi đang học.",
]:
    result = parse_joint_output(text)

    assert result["predicted_error_type"] == 1
    assert result["prediction"] == "Tôi đang học."
    assert result["format_valid"]

print("Đã cập nhật inference và parser.")

Đã cập nhật inference và parser.


In [11]:
import gc
import json
import math
import torch
import transformers

from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)

def train_fold(fold):
    set_seed(SEED)
    gc.collect()
    torch.cuda.empty_cache()

    fold_dir = DATA_DIR / "folds" / f"fold_{fold}"
    run_dir = OUTPUT_DIR / f"fold_{fold}"
    run_dir.mkdir(parents=True, exist_ok=True)

    train_records = read_json(fold_dir / "train.json")
    val_records = read_json(fold_dir / "val.json")

    assert {
        row["source"] for row in train_records
    }.isdisjoint({
        row["source"] for row in val_records
    }), "Input bị trùng giữa train và validation."

    train_dataset = CorrectionDataset(
        train_records,
        tokenizer,
        training=True,
    )
    val_dataset = CorrectionDataset(
        val_records,
        tokenizer,
        training=False,
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

    # Kiểm tra độ dài để không âm thầm cắt input hoặc reference.
    lengths = []

    for dataset in [train_dataset, val_dataset]:
        lengths.extend(
            len(source["input_ids"])
            for source in dataset.sources
        )
        lengths.extend(
            len(target)
            for refs in dataset.targets
            for target in refs
        )

    longest = max(lengths)
    model_limit = model.config.max_position_embeddings

    assert math.ceil(longest / 8) * 8 <= model_limit, (
        f"Có mẫu dài {longest} tokens, "
        f"vượt giới hạn {model_limit}."
    )

    generation_max_length = min(
        model_limit,
        max(256, math.ceil((longest + 32) / 8) * 8),
    )

    model.config.use_cache = False

    collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        pad_to_multiple_of=8,
        label_pad_token_id=-100,
    )

    # Warmup khoảng 10% tổng số optimizer steps.
    n_gpu = max(1, torch.cuda.device_count())
    batches_per_epoch = math.ceil(
        len(train_records) / (TRAIN_BATCH_SIZE * n_gpu)
    )
    steps_per_epoch = math.ceil(
        batches_per_epoch / GRAD_ACCUMULATION
    )
    warmup_steps = max(
        1,
        round(0.1 * EPOCHS * steps_per_epoch),
    )

    args = Seq2SeqTrainingArguments(
        output_dir=str(run_dir / "checkpoints"),
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUMULATION,

        weight_decay=0.01,
        warmup_steps=warmup_steps,
        lr_scheduler_type="linear",
        max_grad_norm=1.0,

        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=5,
        logging_first_step=True,
        disable_tqdm=True,

        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=1,
        save_only_model=True,
    

        fp16=False,
        bf16=False,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={
            "use_reentrant": False
        },

        predict_with_generate=False,
        prediction_loss_only=True,
        dataloader_num_workers=0,
        report_to="none",
        optim="adamw_torch",
        seed=SEED,
        data_seed=SEED,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        data_collator=collator,
        callbacks=[
            ReferenceEpochCallback(train_dataset),
            EarlyStoppingCallback(
                early_stopping_patience=3
            ),
        ],
    )

    print(f"\n===== FOLD {fold} =====", flush=True)
    print("Train:", len(train_records), flush=True)
    print("Validation:", len(val_records), flush=True)
    print("Độ dài lớn nhất:", longest, "tokens", flush=True)
    print("Warmup steps:", warmup_steps, flush=True)
    print("Bắt đầu training...", flush=True)

    from safetensors import safe_open

    resume_checkpoint = None
    candidates = sorted(
        (run_dir / "checkpoints").glob("checkpoint-*"),
        key=lambda path: int(path.name.split("-")[-1]),
        reverse=True,
    )

    for checkpoint in candidates:
        required = [
            checkpoint / "trainer_state.json",
            checkpoint / "optimizer.pt",
            checkpoint / "scheduler.pt",
            checkpoint / "model.safetensors",
        ]

        if not all(path.exists() for path in required):
            continue

        try:
            with safe_open(
                str(checkpoint / "model.safetensors"),
                framework="pt",
                device="cpu",
            ) as saved:
                list(saved.keys())
        except Exception:
            continue

        resume_checkpoint = str(checkpoint)
        break

    if resume_checkpoint:
        print("Tiếp tục từ:", resume_checkpoint, flush=True)
    else:
        print("Không có checkpoint đầy đủ; train lại fold này.", flush=True)

    trainer.train(resume_from_checkpoint=resume_checkpoint)

    # Trainer đã nạp lại checkpoint có validation loss thấp nhất.
    val_loss_metrics = trainer.evaluate()

    model = trainer.model
    model.gradient_checkpointing_disable()
    model.config.use_cache = True

    best_dir = run_dir / "best_model"
    trainer.save_model(str(best_dir))
    tokenizer.save_pretrained(str(best_dir))

    print("Đang sinh câu trên validation...", flush=True)

    raw_predictions = predict_records(
        model,
        val_records,
        batch_size=EVAL_BATCH_SIZE,
        max_length=generation_max_length,
    )

    parsed_outputs = [
        parse_joint_output(text)
        for text in raw_predictions
    ]

    predictions = [
        item["prediction"]
        for item in parsed_outputs
    ]

    metrics = summarize_predictions(
        val_records,
        predictions,
    )
    metrics["fold"] = fold
    metrics["eval_loss"] = val_loss_metrics["eval_loss"]
    metrics["best_checkpoint"] = (
        trainer.state.best_model_checkpoint
    )

    prediction_rows = [
        {
            **row,
            "fold": fold,
            **parsed,
        }
        for row, parsed in zip(
            val_records,
            parsed_outputs,
        )
    ]

    metrics["invalid_label_count"] = sum(
        item["predicted_error_type"] == -1
        for item in parsed_outputs
    )

    metrics["invalid_format_count"] = sum(
        not item["format_valid"]
        for item in parsed_outputs
    )

    write_json(
        run_dir / "val_predictions.json",
        prediction_rows,
    )
    write_json(run_dir / "metrics.json", metrics)
    write_json(
        run_dir / "training_log.json",
        trainer.state.log_history,
    )
    write_json(run_dir / "run_config.json", {
        "model": MODEL_NAME,
        "fold": fold,
        "seed": SEED,
        "epochs_max": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "train_batch_size_per_device": TRAIN_BATCH_SIZE,
        "gradient_accumulation_steps": GRAD_ACCUMULATION,
        "warmup_steps": warmup_steps,
        "generation_max_length": generation_max_length,
        "num_beams": 4,
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
        "validation_loss_reference": "first reference",
    })

    print(json.dumps(
        metrics,
        ensure_ascii=False,
        indent=2,
    ))
    print("Model đã lưu:", best_dir)

    del trainer, model, collator
    gc.collect()
    torch.cuda.empty_cache()

    # Chỉ dọn sau khi đã lưu model và các kết quả.
    checkpoint_dir = run_dir / "checkpoints"

    if checkpoint_dir.exists():
        shutil.rmtree(checkpoint_dir)
        print("Đã dọn checkpoint:", checkpoint_dir)

    return metrics

In [12]:
def summarize_predictions(records, predictions):
    assert len(records) == len(predictions)

    def score(pairs):
        if not pairs:
            return {}

        return {
            "n": len(pairs),
            "exact_match": sum(
                prediction in row["references"]
                for row, prediction in pairs
            ) / len(pairs),
            "unchanged_rate": sum(
                prediction == row["source"]
                for row, prediction in pairs
            ) / len(pairs),
            "keep_input_exact_match": sum(
                row["source"] in row["references"]
                for row, _ in pairs
            ) / len(pairs),
        }

    pairs = list(zip(records, predictions))

    return {
        "overall": score(pairs),
        "by_error_type": {
            str(label): score([
                pair for pair in pairs
                if pair[0]["error_type"] == label
            ])
            for label in range(5)
        },
    }

In [13]:
# Bỏ khôi phục model correction-only; model joint sẽ train ở Cell 9.

In [14]:
import shutil
from pathlib import Path

def show_disk():
    disk = shutil.disk_usage("/kaggle/working")
    print(f"Dung lượng trống: {disk.free / 2**30:.2f} GB")

show_disk()

roots = [
    Path("/kaggle/working/bartpho_vgec"),
    Path("/kaggle/working/bartpho_vgec_joint"),
]

for root in roots:
    if not root.exists():
        continue

    for fold_dir in root.glob("fold_*"):
        checkpoint_dir = fold_dir / "checkpoints"
        best_dir = fold_dir / "best_model"

        weights = (
            list(best_dir.glob("*.safetensors"))
            + list(best_dir.glob("pytorch_model*.bin"))
        )

        completed = (
            (best_dir / "config.json").exists()
            and bool(weights)
            and all(path.stat().st_size > 0 for path in weights)
            and (fold_dir / "val_predictions.json").exists()
            and (fold_dir / "metrics.json").exists()
        )

        if completed and checkpoint_dir.exists():
            shutil.rmtree(checkpoint_dir)
            print("Đã xóa checkpoint trung gian:", checkpoint_dir)

# Chỉ xử lý checkpoint đang ghi dở trong thư mục training hiện tại.
for checkpoint in OUTPUT_DIR.glob("fold_*/checkpoints/checkpoint-*"):
    if not (checkpoint / "trainer_state.json").exists():
        shutil.rmtree(checkpoint)
        print("Đã xóa checkpoint chưa lưu xong:", checkpoint)

show_disk()

Dung lượng trống: 19.50 GB
Dung lượng trống: 19.50 GB


In [15]:
cv_results = []
oof_predictions = []

for fold in range(N_FOLDS):
    run_dir = OUTPUT_DIR / f"fold_{fold}"

    required_files = [
        run_dir / "metrics.json",
        run_dir / "val_predictions.json",
        run_dir / "training_log.json",
        run_dir / "best_model" / "config.json",
    ]

    if all(path.exists() for path in required_files):
        print(f"Fold {fold}: dùng kết quả đã lưu.", flush=True)
        metrics = read_json(run_dir / "metrics.json")
    else:
        metrics = train_fold(fold)

    cv_results.append(metrics)
    oof_predictions.extend(
        read_json(run_dir / "val_predictions.json")
    )

all_records = read_json(DATA_DIR / "processed" / "all.json")

assert len(oof_predictions) == len(all_records)
assert len({row["id"] for row in oof_predictions}) == len(all_records)

oof_predictions.sort(key=lambda row: row["id"])

oof_metrics = summarize_predictions(
    oof_predictions,
    [row["prediction"] for row in oof_predictions],
)

write_json(OUTPUT_DIR / "results.json", cv_results)
write_json(OUTPUT_DIR / "oof_predictions.json", oof_predictions)
write_json(OUTPUT_DIR / "oof_metrics.json", oof_metrics)

print("\nKết quả validation tổng hợp từ 5 folds:")
print(json.dumps(oof_metrics, ensure_ascii=False, indent=2))

pytorch_model.bin:   0%|          | 0.00/526M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/526M [00:00<?, ?B/s]


===== FOLD 0 =====
Train: 352
Validation: 89
Độ dài lớn nhất: 71 tokens
Warmup steps: 16
Bắt đầu training...
Không có checkpoint đầy đủ; train lại fold này.


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'loss': 3.4622, 'grad_norm': 17.8486385345459, 'learning_rate': 0.0, 'epoch': 0.09090909090909091}
{'loss': 3.4501, 'grad_norm': 14.51028823852539, 'learning_rate': 7.5e-06, 'epoch': 0.45454545454545453}
{'loss': 2.2683, 'grad_norm': 8.683440208435059, 'learning_rate': 1.6875e-05, 'epoch': 0.9090909090909091}
{'eval_loss': 1.113538384437561, 'eval_runtime': 1.8687, 'eval_samples_per_second': 47.625, 'eval_steps_per_second': 12.308, 'epoch': 1.0}
{'loss': 1.2445, 'grad_norm': 5.6691155433654785, 'learning_rate': 2.625e-05, 'epoch': 1.3636363636363638}
{'loss': 0.636, 'grad_norm': 3.5456910133361816, 'learning_rate': 2.9395973154362418e-05, 'epoch': 1.8181818181818183}
{'eval_loss': 0.5153911709785461, 'eval_runtime': 1.7857, 'eval_samples_per_second': 49.84, 'eval_steps_per_second': 12.88, 'epoch': 2.0}
{'loss': 0.4519, 'grad_norm': 3.6837270259857178, 'learning_rate': 2.8389261744966442e-05, 'epoch': 2.2727272727272725}
{'loss': 0.4235, 'grad_norm': 2.537883996963501, 'learning_rate':

...

{'train_runtime': 213.0422, 'train_samples_per_second': 24.854, 'train_steps_per_second': 0.845, 'train_loss': 0.6419277588332383, 'epoch': 8.0}
{'eval_loss': 0.3126736581325531, 'eval_runtime': 1.7175, 'eval_samples_per_second': 51.238, 'eval_steps_per_second': 12.809, 'epoch': 8.0}
Đang sinh câu trên validation...
{
  "overall": {
    "n": 88,
    "exact_match": 0.0,
    "unchanged_rate": 0.0,
    "keep_input_exact_match": 0.45454545454545453
  },
  "by_error_type": {
    "0": {
      "n": 40,
      "exact_match": 0.0,
      "unchanged_rate": 0.0,
      "keep_input_exact_match": 1.0
    },
    "1": {
      "n": 29,
      "exact_match": 0.0,
      "unchanged_rate": 0.0,
      "keep_input_exact_match": 0.0
    },
    "2": {
      "n": 6,
      "exact_match": 0.0,
      "unchanged_rate": 0.0,
      "keep_input_exact_match": 0.0
    },
    "3": {
      "n": 6,
      "exact_match": 0.0,
      "unchanged_rate": 0.0,
      "keep_input_exact_match": 0.0
    },
    "4": {
      "n": 7,
      

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'loss': 3.8687, 'grad_norm': 17.973495483398438, 'learning_rate': 0.0, 'epoch': 0.0898876404494382}
{'loss': 3.5667, 'grad_norm': 14.452411651611328, 'learning_rate': 6.666666666666667e-06, 'epoch': 0.449438202247191}
{'loss': 2.3548, 'grad_norm': 9.529742240905762, 'learning_rate': 1.5e-05, 'epoch': 0.898876404494382}
{'eval_loss': 1.068525791168213, 'eval_runtime': 1.7673, 'eval_samples_per_second': 49.794, 'eval_steps_per_second': 12.448, 'epoch': 1.0}
{'loss': 1.163, 'grad_norm': 6.3552632331848145, 'learning_rate': 2.3333333333333336e-05, 'epoch': 1.2696629213483146}
{'loss': 0.7034, 'grad_norm': 4.309333324432373, 'learning_rate': 2.9814814814814815e-05, 'epoch': 1.7191011235955056}
{'eval_loss': 0.4726080596446991, 'eval_runtime': 1.7596, 'eval_samples_per_second': 50.012, 'eval_steps_per_second': 12.503, 'epoch': 2.0}
{'loss': 0.4947, 'grad_norm': 4.199223041534424, 'learning_rate': 2.8888888888888888e-05, 'epoch': 2.0898876404494384}
{'loss': 0.5045, 'grad_norm': 3.8698258399

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


{'train_runtime': 212.944, 'train_samples_per_second': 24.866, 'train_steps_per_second': 0.845, 'train_loss': 0.6355495601843965, 'epoch': 8.0}
{'eval_loss': 0.3758569061756134, 'eval_runtime': 1.7439, 'eval_samples_per_second': 50.461, 'eval_steps_per_second': 12.615, 'epoch': 8.0}
Đang sinh câu trên validation...
{
  "overall": {
    "n": 88,
    "exact_match": 0.0,
    "unchanged_rate": 0.0,
    "keep_input_exact_match": 0.45454545454545453
  },
  "by_error_type": {
    "0": {
      "n": 40,
      "exact_match": 0.0,
      "unchanged_rate": 0.0,
      "keep_input_exact_match": 1.0
    },
    "1": {
      "n": 29,
      "exact_match": 0.0,
      "unchanged_rate": 0.0,
      "keep_input_exact_match": 0.0
    },
    "2": {
      "n": 5,
      "exact_match": 0.0,
      "unchanged_rate": 0.0,
      "keep_input_exact_match": 0.0
    },
    "3": {
      "n": 7,
      "exact_match": 0.0,
      "unchanged_rate": 0.0,
      "keep_input_exact_match": 0.0
    },
    "4": {
      "n": 7,
      "

In [16]:
import json
import math
from pathlib import Path
from collections import Counter

import pandas as pd
from IPython.display import display

BTC_DIR = Path(
    "/kaggle/working/bartpho_vgec_joint/btc_metrics"
)
BTC_DIR.mkdir(parents=True, exist_ok=True)

PRED_PATH = Path(
    "/kaggle/working/bartpho_vgec_joint/oof_predictions.json"
)
DATA_PATH = Path(
    "/kaggle/working/vgec_preprocessed/processed/all.json"
)

def btc_read(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def btc_write(path, value):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            value, f,
            ensure_ascii=False,
            indent=2,
            allow_nan=False,
        )

assert PRED_PATH.exists(), "Chạy cell train đủ 5 fold trước."

btc_rows = btc_read(PRED_PATH)
processed = btc_read(DATA_PATH)
gold_by_id = {row["id"]: row for row in processed}

assert btc_rows
assert len(gold_by_id) == len(processed)
assert len(btc_rows) == len(processed)
assert len({row["id"] for row in btc_rows}) == len(btc_rows)
assert {row["id"] for row in btc_rows} == set(gold_by_id)

for row in btc_rows:
    gold = gold_by_id[row["id"]]

    assert row["source"] == gold["source"]
    assert row["references"] == gold["references"]
    assert row["references"]
    assert row["error_type"] == gold["error_type"]
    assert isinstance(row["prediction"], str)

    row["fold"] = gold["fold"]

btc_rows.sort(key=lambda row: row["id"])

print("Số câu OOF:", len(btc_rows))
print("Các fold:", sorted({row["fold"] for row in btc_rows}))
print(
    "Số câu nhiều reference:",
    sum(len(row["references"]) > 1 for row in btc_rows),
)

Số câu OOF: 441
Các fold: [0, 1, 2, 3, 4]
Số câu nhiều reference: 55


In [28]:
from collections import Counter
from pathlib import Path

# Tách lại từ output gốc, không sinh prediction mới.
for row in btc_rows:
    assert "raw_prediction" in row
    row.update(parse_joint_output(row["raw_prediction"]))

print("Nhãn dự đoán sau sửa:", Counter(
    row["predicted_error_type"]
    for row in btc_rows
))

print("Số output sai format:", sum(
    not row["format_valid"]
    for row in btc_rows
))

for row in btc_rows[:3]:
    print("\nID:", row["id"])
    print("Gold:", row["error_type"])
    print("Predicted:", row["predicted_error_type"])
    print("Câu sửa:", repr(row["prediction"]))

# Cập nhật file OOF để lần sau đọc được kết quả đúng.
btc_write(PRED_PATH, btc_rows)

# Cập nhật prediction từng fold.
root = Path(PRED_PATH).parent

for fold in sorted({row["fold"] for row in btc_rows}):
    fold_path = root / f"fold_{fold}" / "val_predictions.json"

    if fold_path.exists():
        rows = [
            row for row in btc_rows
            if row["fold"] == fold
        ]
        btc_write(fold_path, rows)

print("\nĐã lưu prediction sau khi sửa parser.")

Nhãn dự đoán sau sửa: Counter({0: 235, 1: 199, 3: 4, 4: 2, -1: 1})
Số output sai format: 1

ID: 1
Gold: 1
Predicted: 1
Câu sửa: 'Với đôi tay khéo léo và óc thẩm mĩ tinh tế, người thợ trẻ đã tạo ra được những sản phẩm mành trúc có giá trị.'

ID: 2
Gold: 1
Predicted: 0
Câu sửa: 'Sự phong phú của dân tộc Khơ-me đã góp phần không nhỏ vào kho tàng văn hóa Việt Nam.'

ID: 3
Gold: 1
Predicted: 1
Câu sửa: 'Cuộc sống tối tăm, u uất của chị Dậu đã tố cáo sự thối nát của chế độ thực dân, phong kiến.'

Đã lưu prediction sau khi sửa parser.


In [20]:
from collections import Counter

print("Nhãn gold:", Counter(
    row["error_type"] for row in btc_rows
))

print("Nhãn dự đoán:", Counter(
    row["predicted_error_type"] for row in btc_rows
))

print("Số output sai format:", sum(
    not row["format_valid"] for row in btc_rows
))

# Xem output nguyên bản, trước khi parser tách.
for row in btc_rows[:10]:
    print("\nID:", row["id"])
    print("Gold:", row["error_type"])
    print("Predicted:", row["predicted_error_type"])
    print("Raw:", repr(row["raw_prediction"]))
    print("Câu đã tách:", repr(row["prediction"]))

Nhãn gold: Counter({0: 200, 1: 146, 4: 36, 3: 31, 2: 28})
Nhãn dự đoán: Counter({-1: 441})
Số output sai format: 441

ID: 1
Gold: 1
Predicted: -1
Raw: 'errortype: 1 correctedsentence: Với đôi tay khéo léo và óc thẩm mĩ tinh tế, người thợ trẻ đã tạo ra được những sản phẩm mành trúc có giá trị.'
Câu đã tách: 'errortype: 1 correctedsentence: Với đôi tay khéo léo và óc thẩm mĩ tinh tế, người thợ trẻ đã tạo ra được những sản phẩm mành trúc có giá trị.'

ID: 2
Gold: 1
Predicted: -1
Raw: 'errortype: 0 correctedsentence: Sự phong phú của dân tộc Khơ-me đã góp phần không nhỏ vào kho tàng văn hóa Việt Nam.'
Câu đã tách: 'errortype: 0 correctedsentence: Sự phong phú của dân tộc Khơ-me đã góp phần không nhỏ vào kho tàng văn hóa Việt Nam.'

ID: 3
Gold: 1
Predicted: -1
Raw: 'errortype: 1 correctedsentence: Cuộc sống tối tăm, u uất của chị Dậu đã tố cáo sự thối nát của chế độ thực dân, phong kiến.'
Câu đã tách: 'errortype: 1 correctedsentence: Cuộc sống tối tăm, u uất của chị Dậu đã tố cáo sự thối ná

In [26]:
target = make_joint_target(
    2,
    "Tôi đang học ở trường.",
)

ids = tokenizer(
    text_target=target,
    truncation=False,
)["input_ids"]

decoded = tokenizer.decode(
    ids,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False,
)

print("Target trước tokenize:", repr(target))
print("Target sau decode:", repr(decoded))
print("Parser đọc được:", parse_joint_output(decoded))

Target trước tokenize: 'error_type: 2 corrected_sentence: Tôi đang học ở trường.'
Target sau decode: 'errortype: 2 correctedsentence: Tôi đang học ở trường.'
Parser đọc được: {'raw_prediction': 'errortype: 2 correctedsentence: Tôi đang học ở trường.', 'predicted_error_type': -1, 'prediction': 'errortype: 2 correctedsentence: Tôi đang học ở trường.', 'format_valid': False}


In [29]:
def btc_tokens(text):
    # Token theo khoảng trắng, giữ hoa/thường và dấu câu.
    return text.split()

def btc_ngrams(tokens, n):
    return Counter(
        tuple(tokens[i:i + n])
        for i in range(len(tokens) - n + 1)
    )

def btc_lcs(a, b):
    # LCS: longest common subsequence.
    if len(a) < len(b):
        a, b = b, a

    previous = [0] * (len(b) + 1)

    for x in a:
        current = [0]

        for j, y in enumerate(b, start=1):
            if x == y:
                current.append(previous[j - 1] + 1)
            else:
                current.append(
                    max(previous[j], current[-1])
                )

        previous = current

    return previous[-1]

def btc_score(rows, keep_input=False):
    assert rows

    numerators = [0] * 4
    denominators = [0] * 4
    hypothesis_length = 0
    reference_length = 0
    exact_matches = 0
    rouge_sum = 0.0

    for row in rows:
        source = row["source"]
        hypothesis = (
            source if keep_input else row["prediction"]
        )

        # Chọn cố định, không chọn reference theo prediction.
        reference = row["references"][0]

        # EM: nguyên chuỗi, không strip/normalize.
        exact_matches += hypothesis == reference

        s = btc_tokens(source)
        h = btc_tokens(hypothesis)
        r = btc_tokens(reference)

        hypothesis_length += len(h)
        reference_length += len(r)

        # ROUGE-L F1 = 2*LCS / (len(h) + len(r)).
        lcs_length = btc_lcs(h, r)
        total_length = len(h) + len(r)

        rouge_sum += (
            2 * lcs_length / total_length
            if total_length else 0.0
        )

        # GLEU+, n = 1, 2, 3, 4.
        for n in range(1, 5):
            source_counts = btc_ngrams(s, n)
            hypothesis_counts = btc_ngrams(h, n)
            reference_counts = btc_ngrams(r, n)

            denominators[n - 1] += sum(
                hypothesis_counts.values()
            )

            for gram, count_h in hypothesis_counts.items():
                match_reference = min(
                    count_h, reference_counts[gram]
                )
                match_source = min(
                    count_h, source_counts[gram]
                )

                penalty = max(
                    0, match_source - match_reference
                )

                # Giữ đóng góp âm theo công thức PDF.
                numerators[n - 1] += (
                    match_reference - penalty
                )

    precisions = [
        numerator / denominator if denominator else 0.0
        for numerator, denominator in zip(
            numerators, denominators
        )
    ]

    if hypothesis_length == 0:
        bp = 0.0
    else:
        bp = math.exp(
            min(
                0.0,
                1 - reference_length / hypothesis_length,
            )
        )

    if all(p > 0 for p in precisions):
        gleu = bp * math.exp(
            sum(math.log(p) for p in precisions) / 4
        )
    else:
        gleu = 0.0

    return {
        "n": len(rows),
        "EM": exact_matches / len(rows),
        "GLEU+": gleu,
        "ROUGE-L": rouge_sum / len(rows),
        "GLEU_BP": bp,
        "GLEU_precisions": precisions,
        "GLEU_numerators": numerators,
        "GLEU_denominators": denominators,
    }

# Kiểm tra trường hợp prediction khớp hoàn toàn reference.
test_row = {
    "source": "Tôi đi học vào hôm qua .",
    "prediction": "Tôi đã đi học vào hôm qua .",
    "references": ["Tôi đã đi học vào hôm qua ."],
}

test_score = btc_score([test_row])

for metric in ["EM", "GLEU+", "ROUGE-L"]:
    assert abs(test_score[metric] - 1.0) < 1e-12

print("Đã định nghĩa và kiểm tra các metric.")

Đã định nghĩa và kiểm tra các metric.


In [30]:
btc_report_rows = []
btc_details = {}

def btc_add(scope, system, rows):
    if not rows:
        return

    result = btc_score(
        rows,
        keep_input=(system == "Keep_input"),
    )

    btc_details[f"{scope}/{system}"] = result

    btc_report_rows.append({
        "scope": scope,
        "system": system,
        "n": result["n"],
        "EM": result["EM"],
        "GLEU+": result["GLEU+"],
        "ROUGE-L": result["ROUGE-L"],
    })

# Từng fold.
for fold in sorted({row["fold"] for row in btc_rows}):
    rows = [
        row for row in btc_rows
        if row["fold"] == fold
    ]
    btc_add(f"fold_{fold}", "BARTpho", rows)

# OOF và baseline giữ nguyên input.
error_rows = [
    row for row in btc_rows
    if row["error_type"] != 0
]

for system in ["BARTpho", "Keep_input"]:
    btc_add("OOF_all", system, btc_rows)
    btc_add("OOF_errors_only", system, error_rows)

# Correction score theo nhãn gold, không phải classification F1.
for label in range(5):
    rows = [
        row for row in btc_rows
        if row["error_type"] == label
    ]
    btc_add(f"OOF_gold_type_{label}", "BARTpho", rows)

btc_report = pd.DataFrame(btc_report_rows)
metric_columns = ["EM", "GLEU+", "ROUGE-L"]

view = btc_report.copy()
view[metric_columns] *= 100

print("===== ĐIỂM HIỂN THỊ THANG 0–100 =====")
display(view.round(3))

fold_report = btc_report[
    btc_report["scope"].str.startswith("fold_")
]
fold_summary = fold_report[metric_columns].agg(
    ["mean", "std"]
)

print("===== MEAN / STD GIỮA CÁC FOLD, THANG 0–100 =====")
display((fold_summary * 100).round(3))

btc_report.to_csv(
    BTC_DIR / "correction_report_0_1.csv",
    index=False,
)
fold_summary.to_csv(
    BTC_DIR / "fold_mean_std_0_1.csv",
)
btc_write(
    BTC_DIR / "correction_details.json",
    btc_details,
)
btc_write(
    BTC_DIR / "evaluation_protocol.json",
    {
        "official_scorer": False,
        "tokenization": "whitespace split",
        "case_sensitive": True,
        "exact_match": "raw string equality",
        "reference_selection": "references[0]",
        "gleu": "PDF equation 7; corpus counts; n=1..4",
        "gleu_weights": [0.25] * 4,
        "gleu_smoothing": False,
        "gleu_nonpositive_or_missing_order": "score = 0",
        "rouge_l": "mean sentence LCS F1",
        "rouge_empty_empty": 0,
        "saved_score_scale": "0-1",
        "display_score_scale": "0-100",
        "evaluation_data": (
            "OOF on preprocessed deduplicated inputs; "
            "quarantine excluded"
        ),
    },
)

print("Đã lưu báo cáo tại:", BTC_DIR)

===== ĐIỂM HIỂN THỊ THANG 0–100 =====


,scope,system,n,EM,GLEU+,ROUGE-L
0,fold_0,BARTpho,89,43.820,59.702,85.327
1,fold_1,BARTpho,88,43.182,58.626,85.936
2,fold_2,BARTpho,88,43.182,57.643,84.318
3,fold_3,BARTpho,88,43.182,58.104,87.548
4,fold_4,BARTpho,88,46.591,58.034,85.552
5,OOF_all,BARTpho,441,43.991,58.428,85.735
6,OOF_errors_only,BARTpho,241,5.394,0.000,74.941
7,OOF_all,Keep_input,441,45.351,55.701,87.148
8,OOF_errors_only,Keep_input,241,0.000,0.000,76.483
9,OOF_gold_type_0,BARTpho,200,90.500,97.614,98.742


===== MEAN / STD GIỮA CÁC FOLD, THANG 0–100 =====


,EM,GLEU+,ROUGE-L
mean,43.991,58.422,85.736
std,1.479,0.797,1.176


Đã lưu báo cáo tại: /kaggle/working/bartpho_vgec_joint/btc_metrics


In [31]:
from sklearn.metrics import (
    precision_recall_fscore_support,
    f1_score,
    confusion_matrix,
)

labels = [0, 1, 2, 3, 4]

y_true = [
    row["error_type"]
    for row in btc_rows
]

y_pred = [
    row["predicted_error_type"]
    for row in btc_rows
]

assert all(
    type(x) is int and x in labels
    for x in y_true
)

assert all(
    type(x) is int and x in labels + [-1]
    for x in y_pred
)

p, r, f, support = precision_recall_fscore_support(
    y_true,
    y_pred,
    labels=labels,
    zero_division=0,
)

per_class = pd.DataFrame({
    "error_type": labels,
    "Precision": p,
    "Recall": r,
    "F1": f,
    "support": support,
})

scores = {
    "Macro-F1": float(f1_score(
        y_true,
        y_pred,
        labels=labels,
        average="macro",
        zero_division=0,
    )),
    # Bao gồm nhãn invalid để không bỏ qua lỗi dự đoán.
    "Micro-F1": float(f1_score(
        y_true,
        y_pred,
        labels=labels + [-1],
        average="micro",
        zero_division=0,
    )),
    "invalid_label_count": y_pred.count(-1),
    "invalid_format_count": sum(
        not row["format_valid"]
        for row in btc_rows
    ),
}

cm_labels = labels + [-1]

cm = pd.DataFrame(
    confusion_matrix(
        y_true,
        y_pred,
        labels=cm_labels,
    ),
    index=cm_labels,
    columns=cm_labels,
)

cm.index.name = "gold"
cm.columns.name = "predicted"

display(per_class.round(4))
print("Điểm classification thang 0–1:", scores)
print("Cột -1 trong confusion matrix là nhãn không hợp lệ.")
display(cm)

per_class.to_csv(
    BTC_DIR / "classification_per_class.csv",
    index=False,
)

cm.to_csv(
    BTC_DIR / "confusion_matrix.csv"
)

btc_write(
    BTC_DIR / "classification_scores.json",
    scores,
)

,error_type,Precision,Recall,F1,support
0,0,0.7957,0.9350,0.8598,200
1,1,0.6281,0.8562,0.7246,146
2,2,0.0000,0.0000,0.0000,28
3,3,0.0000,0.0000,0.0000,31
4,4,0.5000,0.0278,0.0526,36


Điểm classification thang 0–1: {'Macro-F1': 0.32740787500986346, 'Micro-F1': 0.7097505668934241, 'invalid_label_count': 1, 'invalid_format_count': 1}
Cột -1 trong confusion matrix là nhãn không hợp lệ.


predicted,0,1,2,3,4,-1
gold,,,,,,
0,187,12,0,0,0,1
1,21,125,0,0,0,0
2,7,19,0,1,1,0
3,7,24,0,0,0,0
4,13,19,0,3,1,0
-1,0,0,0,0,0,0
